# 08 — Independent stroke target validation: sorted endothelium and astrocytic endfeet

This notebook is **independent of the four exploratory MCAO cell/spot datasets**.

**GSE223744**, sorted brain CD31+ endothelial cells, 24h ischemia: **4 independent stroke animals vs 4 naïve controls**. Published values are `expected_counts` (possibly fractional, RSEM output), **not raw integer read counts**. Therefore we use count-sum normalization for descriptive log2CPM, plus an exact animal-label permutation over 70 unique balanced labelings. This is not DESeq2 or a negative-binomial count model. Naïve controls rather than operated shams are a separate confound.

**GSE286075**, early astrocyte endfoot ribosome-bound RNA: **3 paired animals (ipsilateral vs contralateral)** following 2h ischemia and 6h reperfusion. STAR `ReadsPerGene.out.tab` unstranded read counts are used. The paired analysis is descriptive; 3 pairs offer at most 8 sign permutations, so no discovery significance is claimed.

Source annotation is constructed from the GSE223744 EnsemblID_symbol format and transferred to GSE286075 by **base Ensembl gene ID**, preserving unmatched IDs. This is a gene-symbol reference from the independent published experiment, not a genome-version certified mapping.

The candidate list is frozen; host APJ and AMOT are not privileged.


In [ ]:
!pip -q install "pandas>=2.0" "numpy>=1.26" "requests>=2.31" "scipy>=1.12"

from pathlib import Path
import gzip,tarfile,requests,hashlib,itertools,os,re,json
import pandas as pd,numpy as np

ROOT=Path(os.environ.get("IMACS_ROOT", "/content/imacs_stroke_pet_independent" if Path("/content").exists() else "imacs_stroke_pet_independent"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
RAW.mkdir(exist_ok=True,parents=True); OUT.mkdir(exist_ok=True,parents=True)
CAT=pd.read_csv("https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv").drop_duplicates("gene")
def url(a):
 return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{a[:-3]}nnn/{a}/suppl/{a}_RAW.tar"
def fetch(u,p):
 if p.is_file() and p.stat().st_size>100:return p
 with requests.get(u,stream=True,timeout=240) as r:
  r.raise_for_status()
  with p.open("wb") as f:
   for chunk in r.iter_content(1<<20):
    if chunk:f.write(chunk)
 return p
def digest(p):
 h=hashlib.sha256()
 with p.open("rb") as f:
  for b in iter(lambda:f.read(1<<20),b""):h.update(b)
 return h.hexdigest()

sources={}
for a in ("GSE223744","GSE286075"):
 sources[a]=fetch(url(a),RAW/f"{a}_RAW.tar")
pd.DataFrame([{"accession":a,"bytes":p.stat().st_size,"sha256":digest(p),"source":url(a)}
 for a,p in sources.items()]).to_csv(OUT/"08_input_checksums.csv",index=False)


## Four versus four independently purified endothelial transcriptomes

Sample IDs and phenotypes were checked against GEO SOFT:
`GSM6979486-489` are **ischemia**, `GSM6979490-493` are **naïve controls**.
This control design must be distinguished from sham-operated GSE174574.


In [ ]:
phenotypes={**{f"GSM697948{i}":"ischemia" for i in (6,7,8,9)},
            **{f"GSM697949{i}":"control" for i in (0,1,2,3)}}
assert len(phenotypes)==8

ec=[];ensmaps=[]
with tarfile.open(sources["GSE223744"],"r:*") as tar:
 for m in tar:
  if not m.isfile():continue
  gsm=m.name.split("_")[0]
  if gsm not in phenotypes:raise ValueError("Unrecognized GEO sample "+m.name)
  with gzip.GzipFile(fileobj=tar.extractfile(m)) as fd:
   data=pd.read_csv(fd,sep="\t")
  assert {"ID","expected_counts"}.issubset(data.columns),f"Unexpected RSEM columns: {data.columns}"
  annotated=data.ID.astype(str).str.rsplit("_",n=1,expand=True)
  assert annotated.shape[1]==2, "Gene ID_symbol annotation not parsed"
  data["ensembl_base"]=annotated[0].str.split(".").str[0]
  data["gene"]=annotated[1]
  data["expected_counts"]=pd.to_numeric(data["expected_counts"],errors="coerce")
  assert data["expected_counts"].notna().mean()>.99
  assert (data["expected_counts"]>=0).all()
  ensmaps.append(data[["ensembl_base","gene"]].copy())
  data=data.groupby("gene",as_index=False)["expected_counts"].sum()
  total=data.expected_counts.sum()
  assert total>100000, "Sample library unusually small; stop for audit"
  data["log2CPM"]=np.log2(data.expected_counts/total*1e6+0.1)
  data["sample_id"]=gsm
  data["condition"]=phenotypes[gsm]
  ec.append(data[data.gene.isin(CAT.gene)])
  print(gsm,phenotypes[gsm],len(data),"genes",round(total),"estimated read counts")
E=pd.concat(ec,ignore_index=True)
assert E.sample_id.nunique()==8
E.to_csv(OUT/"08_endothelial_four_vs_four_sample_gene_log2CPM.csv",index=False)
A=pd.concat(ensmaps).drop_duplicates()
mapping=A.groupby("ensembl_base")["gene"].nunique()
ambiguous=sorted(mapping[mapping>1].index)
A=A[~A.ensembl_base.isin(ambiguous)]
A=A.drop_duplicates("ensembl_base").set_index("ensembl_base").gene.to_dict()
(OUT/"08_ensembl_annotation_ambiguities.json").write_text(json.dumps({"n_ambiguous":len(ambiguous),"ambiguous_ids":ambiguous[:100]},indent=2))
display(E.head(20))


In [ ]:
def exact_4x4(values,case_mask):
 obs=float(values[case_mask].mean()-values[~case_mask].mean())
 permutations=[]
 for idx in itertools.combinations(range(8),4):
  mask=np.zeros(8,dtype=bool);mask[list(idx)]=True
  permutations.append(float(values[mask].mean()-values[~mask].mean()))
 return obs,float(np.mean(np.abs(permutations)>=abs(obs)-1e-12))

def bh_q(p):
 n=len(p);idx=np.argsort(p)
 q=np.empty(n)
 raw=np.array(p)[idx]*n/np.arange(1,n+1)
 q[idx]=np.minimum.accumulate(raw[::-1])[::-1].clip(0,1)
 return q

rows=[]
for gene in CAT.gene:
 sub=E[E.gene==gene].drop_duplicates("sample_id")
 if sub.sample_id.nunique()!=8:
  rows.append({"gene":gene,"status":"MISSING_IN_ONE_OR_MORE_SAMPLES"})
  continue
 sub=sub.sort_values("sample_id")
 vals=sub.log2CPM.to_numpy()
 case=sub.condition.eq("ischemia").to_numpy()
 assert case.sum()==4 and (~case).sum()==4
 effect,p=exact_4x4(vals,case)
 rows.append({"gene":gene,"status":"TESTABLE","ischemia_minus_naive_log2CPM":effect,
 "permutation_two_sided_p":p,
 "n_ischemia":4,"n_naive":4,
 "all_4_ischemia_above_all_controls":bool(vals[case].min()>vals[~case].max()),
 "all_4_ischemia_below_all_controls":bool(vals[case].max()<vals[~case].min()),
 "naive_mean_log2CPM":float(vals[~case].mean()),
 "ischemia_mean_log2CPM":float(vals[case].mean())})
R=pd.DataFrame(rows)
tested=R.status.eq("TESTABLE")
R.loc[tested,"BH_q_across_frozen_gene_panel"]=bh_q(R.loc[tested,"permutation_two_sided_p"].values)
assert R.loc[tested,"permutation_two_sided_p"].min()>=2/70-1e-12
R.to_csv(OUT/"08_independent_FACS_EC_exact_permutation_4v4.csv",index=False)
display(R.sort_values("permutation_two_sided_p").head(30))


## Astrocyte-endfoot ribosome-bound RNA, paired n=3

The six GEO samples are STAR reads-per-gene files: `CTR1/2/3` are **contralateral** and `Stroke1/2/3` are **ipsilateral**. This is **not sham-vs-stroke**. We take the unstranded gene-count column and match Ensembl IDs to the independent endothelial annotation catalogue.

An astrocyte-specific ribosome-associated RNA result is not a MAO-B protein or PET validation.


In [ ]:
astro=[]
with tarfile.open(sources["GSE286075"],"r:*") as tar:
 for m in tar:
  if not m.isfile():continue
  found=re.search(r"_(CTR|Stroke)([123])ReadsPerGene",m.name)
  assert found is not None, f"Unrecognized endfoot sample: {m.name}"
  condition="contralateral" if found.group(1)=="CTR" else "ipsilateral"
  animal=int(found.group(2))
  with gzip.GzipFile(fileobj=tar.extractfile(m)) as fd:
   d=pd.read_csv(fd,sep="\t",header=None,
     names=["ensembl_id","unstranded","sense","antisense"],dtype={"ensembl_id":str})
  d=d[d.ensembl_id.str.startswith("ENSMUSG")].copy()
  d["ensembl_base"]=d.ensembl_id.str.split(".").str[0]
  d["gene"]=d.ensembl_base.map(A)
  d["unstranded"]=pd.to_numeric(d.unstranded,errors="coerce")
  assert d.unstranded.notna().all()
  lib=d.unstranded.sum()
  assert lib>100000
  d=d[d.gene.isin(CAT.gene)].copy()
  d=d.groupby("gene",as_index=False)["unstranded"].sum()
  d["log2CPM"]=np.log2(d.unstranded/lib*1e6+0.1)
  d["condition"]=condition;d["animal_id"]=animal
  astro.append(d)
  print(condition,animal,"library",int(lib),"matched frozen targets",len(d))
B=pd.concat(astro,ignore_index=True)
assert B.groupby(["animal_id","condition"]).size().shape[0]==6
B.to_csv(OUT/"08_astro_endfoot_paired_sample_gene_log2CPM.csv",index=False)
def pair_rows(gene):
 sub=B[B.gene==gene]
 pivot=sub.pivot_table(index="animal_id",columns="condition",values="log2CPM",aggfunc="mean")
 if not {"ipsilateral","contralateral"}.issubset(pivot.columns):
  return {"gene":gene,"status":"GENE_NOT_MAPPED_IN_ALL_PAIRS"}
 delta=(pivot.ipsilateral-pivot.contralateral).dropna()
 if len(delta)!=3:return {"gene":gene,"status":"INCOMPLETE_PAIRS","n_pairs":len(delta)}
 return {"gene":gene,"status":"TESTABLE","n_pairs":3,
   "mean_paired_ipsilateral_minus_contralateral_log2CPM":float(delta.mean()),
   "n_pairs_ischemia_higher":int((delta>0).sum()),
   "n_pairs_ischemia_lower":int((delta<0).sum()),
   "pair_1_effect":float(delta.loc[1]),"pair_2_effect":float(delta.loc[2]),
   "pair_3_effect":float(delta.loc[3]),
   "independence":"3 paired animals; contrasts inside animal only; no significant p-value possible at two-sided exact p<0.25"}
P=pd.DataFrame([pair_rows(g) for g in CAT.gene])
P.to_csv(OUT/"08_independent_astro_endfoot_paired_3animals.csv",index=False)
display(P)


In [ ]:
text="""# /TEAMRED validity audit — independent sorted-endothelial and astrocyte-endfoot datasets

- GSE223744: 4 ischemia vs 4 **naïve** animals, CD31+ FACS sorted endothelium. The 24h intervention includes surgery-related changes not controlled by sham surgery.
- Input values `expected_counts` contain fractions. They are not raw integer UMI counts. No DESeq2 or inappropriate integer rounding was performed.
- Exact label permutation n=4/4 means minimum possible two-sided p = 2/70 ≈0.0286, and BH q is across frozen genes only.
- GSE286075: 3 PAIRS contralateral vs ipsilateral endfeet, 6h reperfusion (not 24h).
- GSE286075 uses first (unstranded) STAR gene count column; other strand columns are not independent data.
- Ensembl stable-base ID→symbol map from GSE223744 itself, and conflicts are removed. Unmatched genes indicate mapping limitations, not true absence.
- RiboTag captures ribosome-associated RNA, not protein or a PET signal.
- Rat, whole-brain mouse and sorted EC samples are separate experimental models, not combinable effect-size estimates.
- This is candidate adjudication, not proof of tracer accessibility, specificity, affinity, transport or clinical relevance.
"""
(OUT/"08_independent_scientific_audit.md").write_text(text)
print("Complete:",sorted(p.name for p in OUT.iterdir()))
